In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv("../data/processed/demand_features.csv", parse_dates=["date"])

In [3]:
feature_cols = [col for col in df.columns if col not in ["date", "sku_id", "units_sold"]]

In [4]:
model_configs = {
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbosity=0),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
}

all_results = []
trained_models = {}

# Train and evaluate
for model_name, model_template in model_configs.items():
    trained_models[model_name] = {}
    
    for sku in df["sku_id"].unique():
        sku_df = df[df["sku_id"] == sku].sort_values("date").reset_index(drop=True)
        
        # 80/20 time-series split
        split_idx = int(len(sku_df) * 0.8)
        train = sku_df.iloc[:split_idx]
        test = sku_df.iloc[split_idx:]
        
        X_train, y_train = train[feature_cols], train["units_sold"]
        X_test, y_test = test[feature_cols], test["units_sold"]
        
        # Instantiate and train
        model = model_template.__class__(**model_template.get_params())
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        # Metrics
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
        r2 = r2_score(y_test, preds)
        
        all_results.append({
            "model": model_name,
            "sku_id": sku,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2
        })
        trained_models[model_name][sku] = model

In [5]:
results_df = pd.DataFrame(all_results)
summary_df = results_df.groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)
summary_df = summary_df.sort_values("RMSE")

In [6]:
print(summary_df)
results_df.to_csv("../data/processed/tree_based_results.csv", index=False)
summary_df.to_csv("../data/processed/tree_based_summary.csv")
print("Saved tree-based model results.")

                RMSE     MAE    MAPE     R2
model                                      
RandomForest  34.985  27.309  11.621  0.537
LightGBM      35.245  27.472  11.696  0.523
XGBoost       35.602  27.878  11.874  0.514
Saved tree-based model results.
